# ***BAGIAN 1 : Mendapatkan API Key***

In [1]:
!pip install requests python-dotenv --quiet

In [2]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()

TOKEN = os.getenv("TOKEN")
if TOKEN:
    print("Token berhasil dimuat.")
else:
    print("Token Invalid.")

Token berhasil dimuat.


# ***BAGIAN 2: Memanggil API Key***

In [3]:
alamat_api = "https://api.themoviedb.org/3/movie/popular"

headers = {
    "accept"        : "application/json",
    "Authorization" : TOKEN
}

parameter = {
    "language" : "en-US",
    "page"     : 1
}

response = requests.get(alamat_api, headers=headers, params=parameter)
print(f"Status Code: {response.status_code}")

hasil = response.json()
print(f"Jumlah film ditemukan (total_results): {hasil['total_results']}")
print(f"Jumlah film yang dikirim kali ini    : {len(hasil['results'])}")

Status Code: 200
Jumlah film ditemukan (total_results): 20001
Jumlah film yang dikirim kali ini    : 20


In [4]:
film_pertama = hasil["results"][0]
film_pertama

{'adult': False,
 'backdrop_path': '/qeQJx07rK2xm8SD2sJxFKhE7gs0.jpg',
 'genre_ids': [878, 28, 12],
 'id': 969681,
 'title': 'Spider-Man: Brand New Day',
 'original_language': 'en',
 'original_title': 'Spider-Man: Brand New Day',
 'overview': "Fighting crime full-time as Spider-Man in a world that doesn't remember him—and the pressure of seeing his old friends move on without him—sparks a change in Peter Parker he may not have the power to control. But that transformation might also be the only thing that can stop a shocking new threat to the city and those he loves - a powerful villain no one can even see.",
 'popularity': 653.7739,
 'poster_path': '/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg',
 'release_date': '2026-07-29',
 'softcore': False,
 'video': False,
 'vote_average': 7.864,
 'vote_count': 2828}

In [5]:
print("Judul   :", film_pertama["title"])
print("Rilis   :", film_pertama["release_date"])
print("Rating  :", film_pertama["vote_average"])
print("Genre   :", film_pertama["genre_ids"])
print()

alamat_genre = "https://api.themoviedb.org/3/genre/movie/list"
response_genre = requests.get(alamat_genre, headers=headers, params={"language": "en-US"})

daftar_genre = response_genre.json()["genres"]
print("Contoh isi daftar genre:", daftar_genre[:3])

Judul   : Spider-Man: Brand New Day
Rilis   : 2026-07-29
Rating  : 7.864
Genre   : [878, 28, 12]

Contoh isi daftar genre: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 16, 'name': 'Animation'}]


# ***BAGIAN 3: Membungkus Jadi CLass***

In [6]:
class KlienTMDB:

    def __init__(self, TOKEN):
        self.headers = {
            "accept"        : "application/json",
            "Authorization" : TOKEN
        }
        self.alamat_api   = "https://api.themoviedb.org/3/movie/popular"
        self.alamat_genre = "https://api.themoviedb.org/3/genre/movie/list"
        self.peta_genre   = {}

    def _minta(self, alamat, parameter):
        try:
            return requests.get(alamat, headers=self.headers, params=parameter, timeout=20)
        except Exception:
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            try:
                return requests.get(alamat, headers=self.headers, params=parameter, timeout=20)
            except Exception:
                return None

    def _muat_genre(self):
        response = self._minta(self.alamat_genre, {"language": "en-US"})

        if response is None or response.status_code != 200:
            print("Gagal mengambil daftar genre.")
            return

        for genre in response.json()["genres"]:
            self.peta_genre[genre["id"]] = genre["name"]

    def ambil_film(self, halaman):
        if not self.peta_genre:
            self._muat_genre()

        parameter = {
            "language" : "en-US",
            "page"     : halaman
        }

        response = self._minta(self.alamat_api, parameter)

        if response is None or response.status_code != 200:
            status = "tidak ada respon" if response is None else response.status_code
            print(f"Gagal mengambil data halaman {halaman}. Status: {status}")
            return pd.DataFrame()

        daftar_film = response.json()["results"]

        data = []
        for film in daftar_film:
            nama_genre = [self.peta_genre[kode] for kode in film.get("genre_ids", [])
                          if kode in self.peta_genre]

            data.append({
                "ID"          : film.get("id"),
                "Judul"       : film.get("title"),
                "Deskripsi"   : film.get("overview") or None,     # teks kosong dijadikan None
                "Genre"       : ", ".join(nama_genre) if nama_genre else None,
                "Rating"      : film.get("vote_average"),
                "Popularitas" : film.get("popularity"),
                "Tanggal"     : film.get("release_date") or None  # teks kosong dijadikan None
            })

        return pd.DataFrame(data)


print("Class KlienTMDB siap dipakai!")

Class KlienTMDB siap dipakai!


In [7]:
klien = KlienTMDB(TOKEN)

daftar_halaman = list(range(1, 10))

semua_tabel = []

for halaman in daftar_halaman:
    tabel = klien.ambil_film(halaman)
    print(f"Halaman {halaman}: {len(tabel)} film")
    semua_tabel.append(tabel)
    time.sleep(0.5)

df_film = pd.concat(semua_tabel, ignore_index=True)
jumlah_mentah = len(df_film)

print()
print(f"Total film terkumpul: {len(df_film)}")
df_film.head()

Halaman 1: 20 film
Halaman 2: 20 film
Halaman 3: 20 film
Halaman 4: 20 film
Halaman 5: 20 film
Halaman 6: 20 film
Halaman 7: 20 film
Halaman 8: 20 film
Halaman 9: 20 film

Total film terkumpul: 180


,ID,Judul,Deskripsi,Genre,Rating,Popularitas,Tanggal
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,"Science Fiction, Action, Adventure",7.864,653.7739,2026-07-29
1,1423191,Resident Evil,Medical courier Bryan unwittingly finds himsel...,"Horror, Science Fiction, Adventure",7.313,584.5500,2026-09-16
2,1101383,The End of Oak Street,After a mysterious cosmic event rips Oak Stree...,"Science Fiction, Mystery, Thriller",7.011,485.8521,2026-08-12
3,1204680,Coyote vs. Acme,After Acme products fail him one too many time...,"Comedy, Adventure, Family",7.521,409.4076,2026-08-20
4,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...","Adventure, Action, Fantasy",8.016,346.9968,2026-07-15


# ***BAGIAN 4: Membersihkan Data***

In [8]:
print("1. Jumlah sel kosong per kolom:")
print(df_film.isnull().sum())
print()

print("2. Jumlah baris yang kembar (berdasarkan ID):")
print(df_film.duplicated(subset="ID").sum())
print()

print("3. Tipe data setiap kolom:")
print(df_film.dtypes)

1. Jumlah sel kosong per kolom:
ID             0
Judul          0
Deskripsi      0
Genre          1
Rating         0
Popularitas    0
Tanggal        0
dtype: int64

2. Jumlah baris yang kembar (berdasarkan ID):
0

3. Tipe data setiap kolom:
ID               int64
Judul              str
Deskripsi          str
Genre              str
Rating         float64
Popularitas    float64
Tanggal            str
dtype: object


In [9]:
def bersihkan_deskripsi(teks):
    if pd.isna(teks):
        return "Tidak ada deskripsi"
    return teks


def bersihkan_genre(teks):
    if pd.isna(teks):
        return "Tidak Diketahui"
    return teks


deskripsi_kosong = df_film["Deskripsi"].isnull().sum()
genre_kosong     = df_film["Genre"].isnull().sum()

df_film["Deskripsi"] = df_film["Deskripsi"].apply(bersihkan_deskripsi)
df_film["Genre"]     = df_film["Genre"].apply(bersihkan_genre)

jumlah_sebelum = len(df_film)
df_film = df_film.dropna(subset=["Judul"])
judul_dibuang = jumlah_sebelum - len(df_film)
print(f"Baris tanpa judul yang dibuang: {judul_dibuang}")

print()
print("Sel kosong setelah ditangani:")
print(df_film.isnull().sum())

Baris tanpa judul yang dibuang: 0

Sel kosong setelah ditangani:
ID             0
Judul          0
Deskripsi      0
Genre          0
Rating         0
Popularitas    0
Tanggal        0
dtype: int64


In [10]:
jumlah_sebelum = len(df_film)

df_bersih = df_film.drop_duplicates(subset="ID").copy()
kembar_dibuang = jumlah_sebelum - len(df_bersih)

print(f"Jumlah baris sebelum : {jumlah_sebelum}")
print(f"Jumlah baris sesudah : {len(df_bersih)}")
print(f"Baris kembar dibuang : {kembar_dibuang}")

Jumlah baris sebelum : 180
Jumlah baris sesudah : 180
Baris kembar dibuang : 0


In [11]:
def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks, errors="coerce")

print("Tipe data sebelum:", df_bersih["Tanggal"].dtype)

df_bersih["Tanggal"] = df_bersih["Tanggal"].apply(ubah_ke_tanggal)

print("Tipe data sesudah:", df_bersih["Tanggal"].dtype)
df_bersih.head()

Tipe data sebelum: str
Tipe data sesudah: datetime64[us]


,ID,Judul,Deskripsi,Genre,Rating,Popularitas,Tanggal
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,"Science Fiction, Action, Adventure",7.864,653.7739,2026-07-29
1,1423191,Resident Evil,Medical courier Bryan unwittingly finds himsel...,"Horror, Science Fiction, Adventure",7.313,584.5500,2026-09-16
2,1101383,The End of Oak Street,After a mysterious cosmic event rips Oak Stree...,"Science Fiction, Mystery, Thriller",7.011,485.8521,2026-08-12
3,1204680,Coyote vs. Acme,After Acme products fail him one too many time...,"Comedy, Adventure, Family",7.521,409.4076,2026-08-20
4,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...","Adventure, Action, Fantasy",8.016,346.9968,2026-07-15


In [12]:
# Pemeriksaan terakhir sebelum dianggap selesai

print(f"Jumlah baris           : {len(df_bersih)}")
print(f"Sudah lebih dari 100?  : {len(df_bersih) >= 100}")
print(f"Judul masih ada kosong : {df_bersih['Judul'].isnull().sum()}")
print(f"ID masih kembar        : {df_bersih['ID'].duplicated().sum()}")
print(f"Tipe kolom Tanggal     : {df_bersih['Tanggal'].dtype}")

Jumlah baris           : 180
Sudah lebih dari 100?  : True
Judul masih ada kosong : 0
ID masih kembar        : 0
Tipe kolom Tanggal     : datetime64[us]


# ***BAGIAN 5: Simpan Hasil***

In [13]:
df_bersih.to_csv("mini_project_1_Shergy_Diardhan.csv", index=False)
print("Data berhasil disimpan ke file: mini_project_1_Shergy_Diardhan.csv")

df_cek = pd.read_csv("mini_project_1_Shergy_Diardhan.csv")
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: mini_project_1_Shergy_Diardhan.csv
File terbaca kembali: 180 baris, 7 kolom


,ID,Judul,Deskripsi,Genre,Rating,Popularitas,Tanggal
0,969681,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,"Science Fiction, Action, Adventure",7.864,653.7739,2026-07-29
1,1423191,Resident Evil,Medical courier Bryan unwittingly finds himsel...,"Horror, Science Fiction, Adventure",7.313,584.5500,2026-09-16
2,1101383,The End of Oak Street,After a mysterious cosmic event rips Oak Stree...,"Science Fiction, Mystery, Thriller",7.011,485.8521,2026-08-12
3,1204680,Coyote vs. Acme,After Acme products fail him one too many time...,"Comedy, Adventure, Family",7.521,409.4076,2026-08-20
4,1368337,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...","Adventure, Action, Fantasy",8.016,346.9968,2026-07-15


# ***BAGIAN 6: Bahan untuk Slide***

In [14]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)
print(f"Sumber data       : TMDB (themoviedb.org)")
print(f"Halaman dipakai   : {daftar_halaman[0]} sampai {daftar_halaman[-1]}")
print()
print(f"Baris sebelum dibersihkan : {jumlah_mentah}")
print(f"Baris dataset akhir       : {len(df_bersih)}")
print()
print("Class yang dibuat:")
print("  1. KlienTMDB - mengambil data film dari TMDB")
print()
print("Function yang dibuat:")
print("  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda")
print("  2. bersihkan_genre     - mengisi genre kosong dengan penanda")
print("  3. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal")
print()
print("Temuan dari pembersihan data:")
print(f"  Baris kembar dibuang           : {kembar_dibuang}")
print(f"  Baris tanpa judul dibuang      : {judul_dibuang}")
print(f"  Deskripsi kosong diberi penanda: {deskripsi_kosong}")
print(f"  Genre kosong diberi penanda    : {genre_kosong}")
print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data       : TMDB (themoviedb.org)
Halaman dipakai   : 1 sampai 9

Baris sebelum dibersihkan : 180
Baris dataset akhir       : 180

Class yang dibuat:
  1. KlienTMDB - mengambil data film dari TMDB

Function yang dibuat:
  1. bersihkan_deskripsi - mengisi deskripsi kosong dengan penanda
  2. bersihkan_genre     - mengisi genre kosong dengan penanda
  3. ubah_ke_tanggal     - mengubah tulisan tanggal jadi tipe tanggal

Temuan dari pembersihan data:
  Baris kembar dibuang           : 0
  Baris tanpa judul dibuang      : 0
  Deskripsi kosong diberi penanda: 0
  Genre kosong diberi penanda    : 1
